# FX-RL exploration

A clean walkthrough of the pipeline: synthetic data → graph state → environment → baseline comparison.
Training the TD3+GAT agent is done from the CLI (`python scripts/train.py`); this notebook is for inspection only.

The original scratch notebook is archived at `archive_fx_agent_sketch.py`.

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt

from fx_rl.config import Config
from fx_rl.data import generate_gbm_data, make_train_test
from fx_rl.graph import FxGraphBuilder
from fx_rl.env import FXArbitrageEnv
from fx_rl.baselines import default_baselines, run_policy, EqualWeightPolicy
from fx_rl.metrics import compute_metrics

cfg = Config()
cfg.data.n_ticks = 1500  # smaller for a quick notebook
cfg.data.currencies, cfg.data.n_edges

## 1. Synthetic data
Six quoted pairs as independent GBMs with drift; bid/ask via a constant half-spread.

In [ ]:
df = generate_gbm_data(cfg.data, seed=cfg.data.seed)
print(df.shape)
df.head(3)

## 2. Graph state
Each tick becomes a graph: 4 currency nodes, 12 directed edges. Node features are the current/previous portfolio weights; edge features encode the FX rate, spread, volatility, momentum, and a spread-aware arbitrage signal.

In [ ]:
builder = FxGraphBuilder(cfg.data.currencies)
w0 = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float32)  # all USD
g = builder.build(df.iloc[:cfg.env.window_size], current_weights=w0, prev_weights=w0)
print('nodes     :', tuple(g.x.shape))
print('edge_index:', tuple(g.edge_index.shape))
print('edge_attr :', tuple(g.edge_attr.shape))
print('edge feature names: [log_mid_rate, spread_pct, volatility, momentum, arb_signal]')
g.edge_attr[:3]

## 3. Environment rollout
Roll the equal-weight baseline through the environment and plot the NAV trajectory.

In [ ]:
env = FXArbitrageEnv(cfg, df)
res = run_policy(env, EqualWeightPolicy())
m = compute_metrics(res['nav'], res['turnover'], res['transaction_cost'])
print({k: round(v, 4) for k, v in m.items()})

plt.figure(figsize=(10, 4))
plt.plot(res['nav'])
plt.axhline(cfg.env.initial_balance_usd, color='grey', ls='--', alpha=0.5)
plt.title('Equal-weight NAV'); plt.xlabel('step'); plt.ylabel('NAV (USD)')
plt.show()

## 4. Baseline comparison
All baselines share the same dynamics and costs. Note how the random policy is destroyed by transaction costs (high turnover).

In [ ]:
import pandas as pd
_, test_df = make_train_test(cfg.data, train_seed=cfg.data.seed, test_seed=cfg.train.eval_seed, regime_shift=True)
test_env = FXArbitrageEnv(cfg, test_df)
rows = []
for policy in default_baselines():
    r = run_policy(test_env, policy, seed=cfg.train.eval_seed)
    mm = compute_metrics(r['nav'], r['turnover'], r['transaction_cost'])
    mm['policy'] = policy.name
    rows.append(mm)
pd.DataFrame(rows).set_index('policy')[['total_return', 'sharpe', 'max_drawdown', 'avg_turnover', 'total_transaction_cost']].round(4)

## 5. Training the agent
Training is run from the CLI so it is reproducible and logged:

```bash
python scripts/train.py --config configs/default.yaml
python scripts/evaluate.py --model results/models/td3_fx_graph --regime-shift
```

`evaluate.py` adds the trained TD3 agent to the baseline table above and writes a metrics CSV + comparison plot under `results/`.